# Z3rno Python SDK — Quickstart

This notebook walks through the core features of the Z3rno Python SDK:

- **Storing** diverse memory types (semantic, episodic, working, procedural)
- **Recalling** memories with semantic search and filters
- **Graph context** for relationship-aware retrieval
- **Temporal queries** to inspect memory state at any point in time
- **Forgetting** (soft and hard delete)
- **Sessions** for scoped memory operations
- **Audit trail** for compliance and debugging

> **Prerequisites:** A running `z3rno-server` instance is required. Start one locally with
> `docker compose up` or point `base_url` to your hosted deployment. See the
> [deployment guide](https://docs.z3rno.dev/deployment) for details.

In [ ]:
!pip install z3rno

In [ ]:
from z3rno import Z3rnoClient

# Point to your running z3rno-server instance.
# Replace api_key with a real key from the Z3rno dashboard.
client = Z3rnoClient(
    base_url="http://localhost:8000",
    api_key="z3rno_sk_test_...",
)

AGENT_ID = "quickstart-agent-01"

print("Client ready:", client)

## Store Memories

Z3rno supports four memory types:

| Type | Purpose | Example |
|------|---------|---------|
| **semantic** | Long-lived facts and knowledge | "User prefers dark mode" |
| **episodic** | Event records tied to a moment | "User uploaded a CSV at 2:15 PM" |
| **working** | Short-lived scratchpad state | "Current task: summarise Q3 report" |
| **procedural** | How-to knowledge and workflows | "To reset a password, call /auth/reset" |

Let's store 10 diverse memories with relationships between some of them.

In [ ]:
# --- Semantic memories (long-lived facts) ---

m1 = client.store(
    agent_id=AGENT_ID,
    content="User prefers dark mode and high-contrast themes across all applications.",
    memory_type="semantic",
    metadata={"category": "preferences", "source": "onboarding"},
)

m2 = client.store(
    agent_id=AGENT_ID,
    content="User's primary programming language is Python; secondary is TypeScript.",
    memory_type="semantic",
    metadata={"category": "preferences", "source": "conversation"},
)

m3 = client.store(
    agent_id=AGENT_ID,
    content="The company uses PostgreSQL 16 as its primary database with pgvector for embeddings.",
    memory_type="semantic",
    metadata={"category": "architecture", "source": "docs"},
)

# --- Episodic memories (events) ---

m4 = client.store(
    agent_id=AGENT_ID,
    content="User uploaded sales_q3_2025.csv containing 14,320 rows of transaction data.",
    memory_type="episodic",
    metadata={"category": "data_upload", "file": "sales_q3_2025.csv"},
)

m5 = client.store(
    agent_id=AGENT_ID,
    content="User reported a bug: search results are missing after upgrading to v2.4.1.",
    memory_type="episodic",
    metadata={"category": "bug_report", "severity": "high"},
)

m6 = client.store(
    agent_id=AGENT_ID,
    content="User completed the advanced RAG tutorial and rated it 5 stars.",
    memory_type="episodic",
    metadata={"category": "learning", "tutorial": "advanced_rag"},
)

# --- Working memories (scratchpad / short-lived) ---

m7 = client.store(
    agent_id=AGENT_ID,
    content="Current task: generate a quarterly revenue summary from the uploaded CSV.",
    memory_type="working",
    metadata={"category": "active_task"},
)

m8 = client.store(
    agent_id=AGENT_ID,
    content="Draft insight: Q3 revenue is up 12% YoY driven by enterprise tier growth.",
    memory_type="working",
    metadata={"category": "draft_insight"},
)

# --- Procedural memories (how-to) ---

m9 = client.store(
    agent_id=AGENT_ID,
    content=(
        "To connect to the analytics database: "
        "1) Load credentials from Vault at secret/data/analytics. "
        "2) Use asyncpg with SSL mode=verify-full. "
        "3) Set statement_timeout to 30s for safety."
    ),
    memory_type="procedural",
    metadata={"category": "runbook", "system": "analytics_db"},
)

m10 = client.store(
    agent_id=AGENT_ID,
    content=(
        "To deploy a new model version: "
        "1) Tag the commit with semver. "
        "2) Run `make build-image`. "
        "3) Push to ECR. "
        "4) Update the Helm values and apply via ArgoCD."
    ),
    memory_type="procedural",
    metadata={"category": "runbook", "system": "ml_deploy"},
)

# --- Add relationships between related memories ---

# The CSV upload (m4) is related to the active task (m7) and the draft insight (m8)
client.add_relationship(source_id=m4["id"], target_id=m7["id"], relation="triggered")
client.add_relationship(source_id=m7["id"], target_id=m8["id"], relation="produced")

# The database architecture (m3) is related to the analytics runbook (m9)
client.add_relationship(source_id=m3["id"], target_id=m9["id"], relation="context_for")

# The bug report (m5) relates to the database (m3)
client.add_relationship(source_id=m5["id"], target_id=m3["id"], relation="affects")

print(f"Stored {10} memories with relationships.")
print(f"Example memory ID: {m1['id']}")

## Recall Memories

`client.recall()` performs semantic search over stored memories. Results are ranked by
relevance to your natural-language query. You can limit results with `top_k` and narrow
the search with filters like `memory_type`.

In [ ]:
# Broad recall — find the top 5 memories related to user preferences
results = client.recall(
    agent_id=AGENT_ID,
    query="user preferences",
    top_k=5,
)

for r in results:
    print(f"[{r['memory_type']}] (score={r['score']:.3f}) {r['content'][:80]}...")

In [ ]:
# Filtered recall — only semantic memories about databases
results = client.recall(
    agent_id=AGENT_ID,
    query="database",
    memory_type="semantic",
)

for r in results:
    print(f"[{r['memory_type']}] {r['content']}")

## Graph Context

When you stored memories above, some were linked via `add_relationship()`. Passing
`include_graph_context=True` tells Z3rno to traverse those edges and return
**related memories** alongside the direct search hits. This gives your agent richer
context without extra round-trips.

In [ ]:
# Recall with graph context — the search hit plus its graph neighbours
results = client.recall(
    agent_id=AGENT_ID,
    query="uploaded CSV data",
    top_k=3,
    include_graph_context=True,
)

for r in results:
    print(f"[{r['memory_type']}] {r['content'][:90]}")
    # Graph neighbours are returned under the "related" key
    for rel in r.get("related", []):
        print(f"  └─ ({rel['relation']}) {rel['content'][:70]}")
    print()

## Temporal Queries

Z3rno is built on a **bi-temporal data model**. Every memory version is retained, so you
can query the state of memory at any past point in time using the `as_of` parameter.

This is useful for:
- Debugging agent behaviour ("what did it know at 3 PM?")
- Compliance audits
- Reproducing past decisions

In [ ]:
from datetime import datetime, timedelta, timezone

# Recall memories as they existed 1 hour ago
one_hour_ago = datetime.now(timezone.utc) - timedelta(hours=1)

results = client.recall(
    agent_id=AGENT_ID,
    query="revenue summary",
    as_of=one_hour_ago.isoformat(),
)

print(f"Memories visible as of {one_hour_ago.isoformat()}:")
for r in results:
    print(f"  [{r['memory_type']}] {r['content'][:80]}")

In [ ]:
# View the full version history of a specific memory
history = client.get_memory_history(memory_id=m8["id"])

print(f"Version history for memory {m8['id']}:")
for version in history:
    print(f"  v{version['version']}  {version['valid_from']}  {version['content'][:60]}...")

## Forget

Z3rno supports two deletion modes:

- **Soft delete** (`hard=False`, the default) — marks the memory as deleted but retains
  it in the temporal history. It will no longer appear in `recall()` results, but you
  can still access it via `as_of` queries.
- **Hard delete** (`hard=True`) — permanently removes all versions. Use with caution.

In [ ]:
# Soft-delete the working memory for the draft insight
client.forget(memory_id=m8["id"], hard=False)
print(f"Soft-deleted memory {m8['id']}")

# Verify it no longer appears in recall
results = client.recall(
    agent_id=AGENT_ID,
    query="Q3 revenue insight",
    top_k=5,
)

deleted_ids = {r["id"] for r in results}
assert m8["id"] not in deleted_ids, "Memory should be excluded after soft delete"
print("Confirmed: soft-deleted memory is excluded from recall results.")

## Sessions

A **session** groups a sequence of memory operations under a shared context. This is
useful for conversational flows where multiple stores and recalls should be logically
linked. The context manager automatically opens and closes the session.

In [ ]:
# Use a session to group related operations
with client.session(agent_id=AGENT_ID) as sess:
    # Store memories within the session scope
    sess.store(
        content="User asked: 'What were last quarter's top-selling products?'",
        memory_type="episodic",
        metadata={"category": "question"},
    )

    # Recall within the same session — session context is automatically included
    results = sess.recall(query="top-selling products", top_k=3)
    for r in results:
        print(f"[{r['memory_type']}] {r['content'][:80]}")

    sess.store(
        content="Agent answered with a ranked list of 10 products from the Q3 dataset.",
        memory_type="episodic",
        metadata={"category": "answer"},
    )

print(f"Session {sess.session_id} closed.")

## Audit Trail

Every memory operation (store, recall, forget, update) is recorded in an immutable
audit log. Query it to understand what your agent did and when — essential for
debugging, compliance, and observability.

In [ ]:
# Query the audit log for this agent
audit_entries = client.audit(agent_id=AGENT_ID, limit=10)

print(f"Last {len(audit_entries)} audit entries:")
for entry in audit_entries:
    print(
        f"  {entry['timestamp']}  {entry['operation']:8s}  "
        f"memory={entry.get('memory_id', 'N/A')[:8]}...  "
        f"{entry.get('details', '')}"
    )

## Next Steps

You now know the core Z3rno SDK operations. Here are some resources to go deeper:

- **[API Reference](https://docs.z3rno.dev/api)** — Full endpoint documentation
- **[Graph Modelling Guide](https://docs.z3rno.dev/guides/graph)** — Design relationship schemas for your domain
- **[Temporal Queries Deep Dive](https://docs.z3rno.dev/guides/temporal)** — Bi-temporal model explained
- **[Deployment Guide](https://docs.z3rno.dev/deployment)** — Run Z3rno in production with Docker, Kubernetes, or serverless
- **[GitHub](https://github.com/the-ai-project-co/z3rno)** — Source code, issues, and contributions welcome